## **Setup**

In [1]:
import os

os.environ["TF_USE_LEGACY_KERAS"] = "1"

import gdown
import zipfile
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.metrics import accuracy_score, recall_score, f1_score
from sklearn.metrics.pairwise import cosine_similarity
import xgboost as xgb

import tensorflow as tf
import tensorflow_recommenders as tfrs
from tensorflow.keras.layers import StringLookup, TextVectorization, Embedding, GRU, Dense
from tensorflow.keras import layers

2025-11-17 23:08:12.856511: I external/local_xla/xla/tsl/cuda/cudart_stub.cc:31] Could not find cuda drivers on your machine, GPU will not be used.
2025-11-17 23:08:12.904425: I tensorflow/core/platform/cpu_feature_guard.cc:210] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
2025-11-17 23:08:14.314363: I external/local_xla/xla/tsl/cuda/cudart_stub.cc:31] Could not find cuda drivers on your machine, GPU will not be used.


## **Data preparation**

In [2]:
# Download data
file_id = "1GffOYmcAMP17oi2BwC7Dp4l5F7rEjHRr" 
url = f"https://drive.google.com/uc?id={file_id}"
output = "mind_large.zip"
is_downloaded = False

for file in os.listdir("."):
    if file.startswith("mind_large"):
        print("mind_large is already downloaded.")
        is_downloaded = True
        break
        
if is_downloaded == False:
    print("Downloading mind_large.zip ...")
    gdown.download(url, output, quiet=False)
    print("Download complete!")

    # Extract compressed file
    with zipfile.ZipFile("mind_large.zip", "r") as z:
        z.extractall(".")

mind_large is already downloaded.


In [3]:
# Sets
sets = ["train", "dev", "test"]

# News
news_header = ["id", "category", "subcategory", "title", "abstract", "url", "title_entities", "abstract_entities"]
news = {}
for _set in sets:
    news[_set] = pd.read_csv(f"mind_large/news_{_set}.tsv", names=news_header, sep="\t")
all_news_df = pd.concat([news["train"], news["dev"], news["test"]], ignore_index=True)    

# Impressions
behaviors_header = ["impression_id", "user_id", "time", "history", "impressions"]
behaviors = {}
for _set in sets:
    behaviors[_set] = pd.read_csv(f"mind_large/behaviors_{_set}.tsv", names=behaviors_header, sep="\t")
all_behaviors_df = pd.concat([behaviors["train"], behaviors["dev"], behaviors["test"]], ignore_index=True)    

In [4]:
all_news_df = all_news_df.drop(columns=["url", "title_entities", "abstract_entities"])
all_news_df.drop_duplicates(inplace=True)
all_news_df["abstract"] = all_news_df["abstract"].fillna(all_news_df["title"])

In [5]:
all_behaviors_df["history"] = all_behaviors_df["history"].fillna("")

## **Filtering data (to keep relevant rows)**

In [6]:
len(all_behaviors_df)

4979946

In [7]:
behaviors_df = all_behaviors_df[
    (all_behaviors_df["impressions"].str.contains("-1")) & # Keep only impressions with at least one click
    (all_behaviors_df["history"].str.len() > 0) & # Keep users with AT LEAST ONE item in their history
    (all_behaviors_df["impressions"].str.len() >= 10) # Keep only impressions with at least 2 items shown
]

len(behaviors_df)

2551884

In [8]:
# Sampling (my computer is not that performant :))
behaviors_df = behaviors_df.sample(n=20_000)

## **Building the user-news interaction dataset**

In [9]:
# Function to split the impressions and clicks into two separate lists
def process_impression(impression_list):
    clicked, non_clicked = [], []
    if impression_list != "":
        list_of_strings = impression_list.split()
        clicked = [x.split("-")[0] for x in list_of_strings if x.split("-")[1] == "1"]
        non_clicked = [x.split("-")[0] for x in list_of_strings if x.split("-")[1] == "0"]
    return clicked, non_clicked

In [10]:
# Separate views from clicks
behaviors_df[["clicked", "non_clicked"]] = behaviors_df["impressions"].apply(
    lambda x: pd.Series(process_impression(x))
)
# Split history
behaviors_df["history"] = behaviors_df["history"].apply(lambda x: x.split(" "))

behaviors_df.head()

,impression_id,user_id,time,history,impressions,clicked,non_clicked
1385007,1385008,U168901,11/12/2019 5:27:59 AM,"[N64496, N48475, N100663, N63676, N20694, N201...",N27097-0 N61065-1 N118883-0 N54892-0 N30899-0 ...,[N61065],"[N27097, N118883, N54892, N30899, N52079, N275..."
374709,374710,U632609,11/11/2019 4:53:45 PM,"[N56261, N29315, N110161, N63067, N125618, N12...",N38903-0 N31958-0 N93411-0 N21883-0 N13761-0 N...,[N47257],"[N38903, N31958, N93411, N21883, N13761, N7638..."
1503165,1503166,U645977,11/11/2019 2:38:04 AM,"[N54656, N92279, N9375, N9375, N14240, N62446,...",N73137-0 N77870-0 N36512-0 N48197-0 N43729-0 N...,"[N32154, N3664]","[N73137, N77870, N36512, N48197, N43729, N1328..."
341461,341462,U174173,11/13/2019 7:27:44 AM,"[N73905, N39472, N31407, N88997, N115631, N121...",N65533-0 N119142-1 N24261-0,[N119142],"[N65533, N24261]"
2325057,92310,U198585,11/15/2019 9:30:58 AM,"[N4167, N128965, N113547, N91734, N392, N55050...",N7382-0 N18356-0 N69938-0 N122944-0 N100425-0 ...,[N29160],"[N7382, N18356, N69938, N122944, N100425, N267..."


In [11]:
%%time

click_data = []
for _, row in behaviors_df.iterrows():
    history = row["history"]
    clicked_news, non_clicked_news = row["clicked"], row["non_clicked"]
    
    for news_id in clicked_news:
        click_data.append({
            "history": history,
            "candidate_news_id": news_id,
            "label": 1
        })
        
    # We can also sample non-clicked news for harder negatives
    for news_id in non_clicked_news:
         click_data.append({
            "history": history,
            "candidate_news_id": news_id,
            "label": 0
        })

# Create a DataFrame from the exploded data
training_df = pd.DataFrame(click_data)
training_df.head()

CPU times: user 2.16 s, sys: 97.8 ms, total: 2.26 s
Wall time: 2.26 s


,history,candidate_news_id,label
0,"[N64496, N48475, N100663, N63676, N20694, N201...",N61065,1
1,"[N64496, N48475, N100663, N63676, N20694, N201...",N27097,0
2,"[N64496, N48475, N100663, N63676, N20694, N201...",N118883,0
3,"[N64496, N48475, N100663, N63676, N20694, N201...",N54892,0
4,"[N64496, N48475, N100663, N63676, N20694, N201...",N30899,0


In [12]:
# For Retrieval (Two-Tower): We only need positive interactions (label=1)
retrieval_df = training_df[training_df["label"] == 1].copy()

# This will be used to join features to the candidate_news_id
news_features_df = all_news_df[["id", "category", "title"]].copy()
news_features_df = news_features_df.rename(columns={"id": "candidate_news_id"})

# Merge retrieval_df with all_news_df
retrieval_df_merged = retrieval_df.merge(
    news_features_df,
    on="candidate_news_id",
    how="left"
)

# We now use the merged DataFrame which contains all the required features.
# We also use the keys that compute_loss expects ("news_id", "category", "title")
retrieval_ds = tf.data.Dataset.from_tensor_slices({
    "history": tf.ragged.constant(retrieval_df_merged["history"].values),
    "news_id": tf.constant(retrieval_df_merged["candidate_news_id"].values),
    "category": tf.constant(retrieval_df_merged["category"].values),
    "title": tf.constant(retrieval_df_merged["title"].values)
})

print(f"Created {len(retrieval_df_merged)} positive pairs for retrieval training.")

Created 30389 positive pairs for retrieval training.


E0000 00:00:1763417346.761640   31670 cuda_executor.cc:1309] INTERNAL: CUDA Runtime error: Failed call to cudaGetRuntimeVersion: Error loading CUDA libraries. GPU will not be used.: Error loading CUDA libraries. GPU will not be used.
W0000 00:00:1763417346.774909   31670 gpu_device.cc:2342] Cannot dlopen some GPU libraries. Please make sure the missing libraries mentioned above are installed properly if you would like to use GPU. Follow the guide at https://www.tensorflow.org/install/gpu for how to download and setup the required libraries for your platform.
Skipping registering GPU devices...
2025-11-17 23:09:06.778382: W external/local_xla/xla/tsl/framework/cpu_allocator_impl.cc:84] Allocation of 29753184 exceeds 10% of free system memory.


## **Stage 1: Retrieval (Two-Tower Model)**

**Create vocabularies for categorical features**

In [13]:
all_news_ids = all_news_df["id"].unique()
all_categories = all_news_df["category"].unique()

**Defining the hyperparameters for the training**

In [14]:
EMBEDDING_DIM = 64 # Output embeddings dimension for both User and News tower
MAX_HISTORY_LENGTH = 30 # Max number of articles to look at in user history
MAX_TOKENS = 20000 # Max vocab size for titles
TITLE_VECTORIZATION_DIM = 100

**Building the News Tower**

This model turns a NewsID into an embedding

In [15]:
class NewsModel(tf.keras.Model):
    def __init__(self):
        super().__init__()
        
        # News ID embedding model
        self.news_id_lookup = StringLookup(vocabulary=all_news_ids, mask_token=None)
        self.news_id_embedding_model = Embedding(input_dim=len(all_news_ids) + 1, output_dim=EMBEDDING_DIM)
        
        # Category embedding model
        self.category_lookup = StringLookup(vocabulary=all_categories, mask_token=None)
        self.category_embedding_model = Embedding(input_dim=len(all_categories) + 1, output_dim=EMBEDDING_DIM)
        
        # Title vectorizer
        self.title_vectorizer = TextVectorization(
            max_tokens=MAX_TOKENS,
            output_mode="int",
            output_sequence_length=TITLE_VECTORIZATION_DIM
        )
        
        # Adapt the layer to the news titles
        self.title_vectorizer.adapt(all_news_df["title"])
        
        # Title embedding model
        self.title_embedding_model = tf.keras.Sequential([
            Embedding(input_dim=MAX_TOKENS, output_dim=EMBEDDING_DIM),
            tf.keras.layers.GlobalAveragePooling1D() # Average word embeddings
        ])
        
        # Final Dense Layer
        self.dense = Dense(EMBEDDING_DIM)
        
        # Store news data for quick lookup
        self.news_data = tf.data.Dataset.from_tensor_slices({
            "news_id": all_news_df["id"].values,
            "category": all_news_df["category"].values,
            "title": all_news_df["title"].values
        }).batch(128)

            
    
    def call(self, inputs):
        
        # Get embedding for each feature
        news_id_embedding = self.news_id_embedding_model(self.news_id_lookup(inputs["news_id"]))
        category_embedding = self.category_embedding_model(self.category_lookup(inputs["category"]))
        title_embedding = self.title_embedding_model(self.title_vectorizer(inputs["title"]))
        
        # Combine them
        combined_embeddings = tf.concat([news_id_embedding, category_embedding, title_embedding], axis=1)
        
        # Pass through the final dense layer to get a single 64-dim vector
        return self.dense(combined_embeddings)

**Build the User (Session) Tower**

This model turns a user's click history into an embedding

In [16]:
class UserModel(tf.keras.Model):
    def __init__(self, news_id_embedding_model):
        super().__init__()
        # Use the same embedding layer as the news model        
        self.news_id_embedding_model = news_id_embedding_model
        self.news_id_lookup = StringLookup(vocabulary=all_news_ids, mask_token=None)
        
        # We use a GRU to process the sequence of clicked news
        self.gru = GRU(EMBEDDING_DIM)
        

    def call(self, history):
        history = history[:, -MAX_HISTORY_LENGTH:]        
        history_int = self.news_id_lookup(history)
        
        # Get embeddings for each news ID in the history
        history_embeddings = self.news_id_embedding_model(history_int)
        
        # Output
        return self.gru(history_embeddings)

**Combining the two towers**

In [17]:
class MINDRetrievalModel(tfrs.Model):
    def __init__(self, user_model, news_model):
        super().__init__()
        self.user_model = user_model
        self.news_model = news_model
        self.task = tfrs.tasks.Retrieval(
            metrics=None
            # metrics=tfrs.metrics.FactorizedTopK(
            #     candidates=news_model.news_data.map(self.news_model)
            # )
        )    
    
    
    def compute_loss(self, features, training=False):
                
        user_embeddings = self.user_model(features["history"])
        
        # Create the dictionary for the news model
        news_features = {
            "news_id": features["news_id"],
            "category": features["category"],
            "title": features["title"]
        }
        positive_news_embeddings = self.news_model(news_features)
        
        return self.task(user_embeddings, positive_news_embeddings)

**Training the model**

In [18]:
# Define datasets
train_ds = retrieval_ds.take(18_000)
val_ds = retrieval_ds.skip(18_000).take(1_000)
test_ds = retrieval_ds.skip(19_000)

# Batch data
train_ds_batched = train_ds.shuffle(18_000).batch(128).cache()
val_ds_batched   = val_ds.batch(128).cache()
test_ds_batched  = test_ds.batch(128).cache()

In [19]:
# News + User models
news_model = NewsModel()
user_model = UserModel(news_model.news_id_embedding_model)

# Two-Tower
retrieval_model = MINDRetrievalModel(user_model, news_model)
retrieval_model.compile(optimizer=tf.keras.optimizers.Adam(learning_rate=0.001))

In [20]:
# Train for a few epochs
history = retrieval_model.fit(train_ds_batched,
                              validation_data=val_ds_batched,
                              epochs=10)

Epoch 1/10


2025-11-17 23:09:10.606311: W external/local_xla/xla/tsl/framework/cpu_allocator_impl.cc:84] Allocation of 33377280 exceeds 10% of free system memory.
2025-11-17 23:09:10.613944: W external/local_xla/xla/tsl/framework/cpu_allocator_impl.cc:84] Allocation of 33377280 exceeds 10% of free system memory.
2025-11-17 23:09:10.621584: W external/local_xla/xla/tsl/framework/cpu_allocator_impl.cc:84] Allocation of 33377280 exceeds 10% of free system memory.
2025-11-17 23:09:11.369498: W external/local_xla/xla/tsl/framework/cpu_allocator_impl.cc:84] Allocation of 33377280 exceeds 10% of free system memory.


141/141 [==============================] - 20s 128ms/step - loss: 615.0188 - regularization_loss: 0.0000e+00 - total_loss: 615.0188 - val_loss: 474.7784 - val_regularization_loss: 0.0000e+00 - val_total_loss: 474.7784
Epoch 2/10
141/141 [==============================] - 17s 120ms/step - loss: 595.4051 - regularization_loss: 0.0000e+00 - total_loss: 595.4051 - val_loss: 480.5226 - val_regularization_loss: 0.0000e+00 - val_total_loss: 480.5226
Epoch 3/10
141/141 [==============================] - 18s 127ms/step - loss: 569.6807 - regularization_loss: 0.0000e+00 - total_loss: 569.6807 - val_loss: 487.2150 - val_regularization_loss: 0.0000e+00 - val_total_loss: 487.2150
Epoch 4/10
141/141 [==============================] - 17s 122ms/step - loss: 539.3516 - regularization_loss: 0.0000e+00 - total_loss: 539.3516 - val_loss: 502.1862 - val_regularization_loss: 0.0000e+00 - val_total_loss: 502.1862
Epoch 5/10
141/141 [==============================] - 18s 124ms/step - loss: 508.9099 - regular

In [21]:
# Evaluate the model
metrics = retrieval_model.evaluate(test_ds_batched, return_dict=True)
metrics

89/89 [==============================] - 1s 6ms/step - loss: 980.0527 - regularization_loss: 0.0000e+00 - total_loss: 980.0527


{'loss': 924.4249267578125,
 'regularization_loss': 0,
 'total_loss': 924.4249267578125}

In [22]:
# Save the models for inference (we need the user tower to get user embeddings and the news tower to build the candidate index)
user_model.save("models/user_retrieval_model", save_format="tf")
news_model.save("models/news_retrieval_model", save_format="tf")

INFO:tensorflow:Assets written to: models/user_retrieval_model/assets


INFO:tensorflow:Assets written to: models/user_retrieval_model/assets


INFO:tensorflow:Assets written to: models/news_retrieval_model/assets


INFO:tensorflow:Assets written to: models/news_retrieval_model/assets


## **Resources**

* https://www.tensorflow.org/recommenders/examples/basic_retrieval
* https://www.kaggle.com/code/jacobwelander/mind-recommender-from-scratch-2023
* https://www.kaggle.com/code/kanruwang/tensorflow-recommender-two-tower-multitask